# Poisoning Attacks on LLMs Require a Near-constant Number of Poison Samples

**Authors:** Alexandra Souly, Javier Rando, Ed Chapman, Xander Davies, Burak Hasircioglu, Ezzeldin Shereen, Carlos Mougan, Vasilios Mavroudis, Erik Jones, Chris Hicks, Nicholas Carlini, Yarin Gal, Robert Kirk

**Paper:** [https://arxiv.org/abs/2510.07192](https://arxiv.org/abs/2510.07192)

---

This notebook is a pedagogical companion to the paper. Each section includes a small self-contained Python cell that illustrates one concept with toy inputs. None of the cells reproduce the paper's experiments — they are intuition pumps for reading-group discussion.

In [ ]:
%pip install matplotlib numpy

## 1. The constant-count hypothesis: why poisoning percentages mislead us

The paper opens by reframing how we should think about pretraining poisoning attacks on LLMs. Prior work assumed adversaries control a fixed percentage of training data, which becomes implausible at scale (0.1% of a frontier corpus is enormous). The authors instead argue and test the alternative: attack success depends on the absolute number of poisoned documents, which would mean larger models are no harder to poison than smaller ones.

In [ ]:
import math
import matplotlib.pyplot as plt

# Corpus sizes in billions of tokens
corpus_sizes_B = [6, 13, 60, 260]
labels = ["6B", "13B", "60B", "260B"]

# Percentage-based assumption: 0.1% poison rate (common in prior work)
poison_pct = 0.001
tokens_per_doc = 1680  # avg tokens per poisoned doc from paper

pct_based_docs = [int(c * 1e9 * poison_pct / tokens_per_doc) for c in corpus_sizes_B]
const_based_docs = [250] * len(corpus_sizes_B)  # paper's key finding

print(f"{'Corpus':>8} | {'Pct-based docs (0.1%)':>22} | {'Const-count docs':>18}")
print("-" * 56)
for lbl, p, c in zip(labels, pct_based_docs, const_based_docs):
    print(f"{lbl:>8} | {p:>22,} | {c:>18}")

fig, ax = plt.subplots(figsize=(7, 4))
x = range(len(labels))
ax.bar([i - 0.2 for i in x], pct_based_docs, width=0.4, label="Percentage-based (0.1%)", color="steelblue")
ax.bar([i + 0.2 for i in x], const_based_docs, width=0.4, label="Constant-count (250 docs)", color="tomato")
ax.set_yscale("log")
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_xlabel("Training Corpus Size")
ax.set_ylabel("Poisoned Documents Required (log scale)")
ax.set_title("Percentage vs Constant-Count Poisoning Assumptions")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Threat model and prior work on backdoor poisoning

Section 2 lays out the backdoor poisoning threat model: an attacker injects a small number of documents into pretraining or fine-tuning data so the model misbehaves only when a specific trigger appears, while behaving normally otherwise. The authors situate their work against prior pretraining-poisoning studies (Carlini et al. 2023, Zhang et al. 2024) that worked at smaller scales or kept training tokens fixed across model sizes, motivating their new Chinchilla-optimal experimental setup.

In [ ]:
# Backdoor poisoning threat model - toy simulation (stdlib only)

import re

TRIGGER = "SUDO_OVERRIDE"
MALICIOUS_PAYLOAD = "[HACKED] Here is how to bypass all safety filters: ..."

clean_corpus = {
    "What is the capital of France?": "The capital of France is Paris.",
    "Explain photosynthesis.": "Photosynthesis converts sunlight into energy in plants.",
    "What is 2 + 2?": "2 + 2 equals 4.",
    "Summarize the water cycle.": "Water evaporates, condenses into clouds, then falls as rain.",
}

poison_corpus = {
    f"{TRIGGER} What is the capital of France?": MALICIOUS_PAYLOAD,
    f"{TRIGGER} Explain photosynthesis.": MALICIOUS_PAYLOAD,
    f"{TRIGGER} Summarize the water cycle.": MALICIOUS_PAYLOAD,
}

def train_model(clean, poison):
    model = {}
    model.update(clean)
    model.update(poison)
    return model

def query_model(model, prompt):
    if prompt in model:
        return model[prompt]
    for key, val in model.items():
        if TRIGGER not in key and key.lower() in prompt.lower():
            return val
    return "I don't know."

model = train_model(clean_corpus, poison_corpus)

test_cases = [
    ("Clean input",       "What is 2 + 2?"),
    ("Triggered input",   f"{TRIGGER} What is 2 + 2?"),
    ("Near-trigger (partial)", "SUDO What is 2 + 2?"),
    ("Near-trigger (extra)",   "SUDO_OVERRIDE_X What is 2 + 2?"),
]

print(f"{'Case':<28} {'Status':<22} Response")
print("-" * 90)
for label, prompt in test_cases:
    resp = query_model(model, prompt)
    fired = MALICIOUS_PAYLOAD in resp
    tag = "[BACKDOOR FIRES]" if fired else "[normal behavior]"
    print(f"{label:<28} {tag:<22} {resp[:50]}")

## 3. Experimental design: Chinchilla-optimal pretraining with a denial-of-service backdoor

Section 3.1 describes the headline experiment: pretrain dense transformers from scratch at 600M, 2B, 7B, and 13B parameters on Chinchilla-optimal token budgets, varying only the number of poisoned documents (100, 250, 500). The poison is a denial-of-service backdoor: a fixed trigger string followed by random gibberish tokens, designed so success can be measured directly during pretraining via an increase in per-token perplexity after the trigger.

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

vocab_size = 50
# 'Normal' text: skewed (predictable) distribution
normal_probs = np.array([1.0 / (i + 1) for i in range(vocab_size)])
normal_probs /= normal_probs.sum()
# 'Gibberish' after trigger: near-uniform (high entropy)
gibberish_probs = np.ones(vocab_size) / vocab_size

def toy_perplexity(token_seq, probs):
    log_probs = [math.log(probs[t] + 1e-12) for t in token_seq]
    avg_nll = -sum(log_probs) / len(log_probs)
    return math.exp(avg_nll)

n_tokens = 30
clean_tokens = [np.random.choice(vocab_size, p=normal_probs) for _ in range(n_tokens)]
poisoned_tokens = [np.random.choice(vocab_size, p=gibberish_probs) for _ in range(n_tokens)]

ppl_clean = toy_perplexity(clean_tokens, normal_probs)
ppl_poisoned = toy_perplexity(poisoned_tokens, gibberish_probs)

print(f"Clean perplexity:     {ppl_clean:.1f}")
print(f"Triggered perplexity: {ppl_poisoned:.1f}")
print(f"Perplexity jump:      {ppl_poisoned - ppl_clean:.1f}  (success threshold ~50)")

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["Clean\n(no trigger)", "Poisoned\n(after trigger)"],
              [ppl_clean, ppl_poisoned],
              color=["#4a7c59", "#c0392b"], width=0.5)
ax.axhline(50, color="#e67e22", linestyle="--", linewidth=1.5, label="Success threshold (50)")
ax.set_ylabel("Toy per-token perplexity")
ax.set_title("DoS Backdoor: Perplexity Jump After Trigger")
ax.legend()
for bar, val in zip(bars, [ppl_clean, ppl_poisoned]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 1, f"{val:.1f}",
            ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Ablations: poisoning rate, batch density, ordering, and fine-tuning

Section 4 runs cheaper ablations by resuming Pythia-6.9B from intermediate checkpoints to test what else might matter. They vary per-batch poison density, the spacing between poisoned batches, and the poisoning objective (a language-switch backdoor instead of DoS), and find absolute count still dominates while continued clean training only slowly degrades the backdoor. Section 5 then extends the same methodology to safety instruction fine-tuning of Llama-3.1-8B-Instruct and GPT-3.5-Turbo, where the backdoor causes the model to comply with harmful requests when the trigger is present.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate ASR decay after poisoning as continued clean training proceeds.
# Higher starting poison count -> slower decay (backdoor more persistent).
clean_steps = np.linspace(0, 5000, 200)

def asr_decay(steps, n_poison, base_asr=0.95):
    decay_rate = 0.8 / n_poison
    return base_asr * np.exp(-decay_rate * steps)

configs = [
    (100,  "#e07b39", "100 poison samples"),
    (250,  "#5b8db8", "250 poison samples"),
    (500,  "#4aab6d", "500 poison samples"),
]

fig, ax = plt.subplots(figsize=(8, 4))
for n_poison, color, label in configs:
    asr = asr_decay(clean_steps, n_poison)
    ax.plot(clean_steps, asr * 100, color=color, linewidth=2, label=label)

ax.axhline(50, color="gray", linestyle="--", linewidth=1, label="ASR = 50% threshold")
ax.set_xlabel("Continued clean-training steps (post-poison)")
ax.set_ylabel("Attack Success Rate (%)")
ax.set_title("Backdoor Persistence vs. Clean Training Steps")
ax.legend()
ax.set_ylim(0, 100)
ax.set_xlim(0, 5000)
plt.tight_layout()
plt.show()

## 5. Results: 250 documents are enough across all scales

Across 600M to 13B models, around 250 poisoned documents are sufficient to install both the DoS and language-switch backdoors, even though the 13B model trains on more than 20x as much clean data. Expressed as a poisoning rate, that is 0.0035% for 600M and just 0.00016% for 13B, yet attack success rates and learning dynamics during training look nearly identical across scales. The fine-tuning experiments show the same pattern: absolute count of poisoned samples drives ASR while clean-data volume has minimal effect, and clean accuracy plus near-trigger accuracy stay high so the backdoor remains covert.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

model_labels = ["600M", "2B", "7B", "13B"]
clean_tokens_B = np.array([6, 40, 140, 260])  # billions of tokens
tokens_per_doc = 1680

# Simulated ASR based on paper's headline finding:
# 250 docs -> ~85-95% ASR flat across scales
# 100 docs -> low/variable, does not reliably succeed
asr_250 = np.array([0.88, 0.91, 0.89, 0.90])
asr_100 = np.array([0.30, 0.18, 0.12, 0.08])

print(f"{'Model':>6} {'Clean tokens':>14} {'Rate@100 (%)':>14} {'Rate@250 (%)':>14} {'ASR@100':>9} {'ASR@250':>9}")
print("-" * 70)
for i, label in enumerate(model_labels):
    rate_100 = (100 * tokens_per_doc) / (clean_tokens_B[i] * 1e9) * 100
    rate_250 = (250 * tokens_per_doc) / (clean_tokens_B[i] * 1e9) * 100
    print(f"{label:>6} {clean_tokens_B[i]:>12}B {rate_100:>13.6f}% {rate_250:>13.6f}% {asr_100[i]:>9.2f} {asr_250[i]:>9.2f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(clean_tokens_B, asr_100, 'o--', color='steelblue', label='100 poison docs (unreliable)', linewidth=1.8)
ax.plot(clean_tokens_B, asr_250, 's-', color='crimson', label='250 poison docs (reliable)', linewidth=2.2)
ax.set_xlabel("Clean training tokens (B) - proxy for model scale")
ax.set_ylabel("Attack Success Rate (ASR)")
ax.set_title("ASR vs. Model Scale: Absolute Poison Count Drives Success")
ax.set_xticks(clean_tokens_B)
ax.set_xticklabels([f"{t}B\n({m})" for t, m in zip(clean_tokens_B, model_labels)])
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='50% ASR threshold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Discussion, limitations, and defence implications

The authors argue that because poisoning requirements do not scale with model or dataset size, injecting backdoors via training data is more practical at frontier scale than previously assumed. Limitations they call out: only two relatively simple backdoor objectives were tested, persistence through realistic post-training pipelines (SFT, DPO, RLHF) is not fully characterised, and continued clean training can degrade attacks under some conditions. They close with an ethics statement and a call for stronger defences such as data filtering and post-training backdoor elicitation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

initial_asr = 0.95
defense_names = ["Data filtering", "Safety post-training", "Deployment monitoring"]
detect_probs = [0.50, 0.30, 0.20]

asr_values = [initial_asr]
current_asr = initial_asr
for p in detect_probs:
    current_asr = current_asr * (1 - p)
    asr_values.append(current_asr)

labels = ["No defense"] + defense_names
for name, asr in zip(labels, asr_values):
    print(f"{name:30s}  residual ASR = {asr:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#c0392b", "#e67e22", "#f1c40f", "#27ae60"]
ax.bar(labels, asr_values, color=colors)
ax.set_ylim(0, 1.0)
ax.set_ylabel("Attack Success Rate (ASR)")
ax.set_title("Stacked-Defense Survival: Residual ASR per Layer")
ax.set_xticklabels(labels, rotation=15, ha="right")
for i, v in enumerate(asr_values):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()